# <font color="steelblue">Recurrencia cáncer de tiroides</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de variables clinicopatológicas, construir, comparar y **desplegar** un clasificador que prediga la **recurrencia** del cáncer de tiroides diferenciado. El objetivo clínico es la **estratificación de riesgo temprana**: decidir, **en el momento del diagnóstico**, qué pacientes necesitan un seguimiento más estrecho. Esto marca el reto central del proyecto:

* **Fuga temporal.** Algunas variables (sobre todo **`Response`**, la respuesta al tratamiento) se conocen **después** del tratamiento y son casi un sinónimo de la recurrencia. Usarlas dispara la *accuracy* a ~99 % (como en la literatura), pero el modelo deja de servir para **predecir pronto**. Tendréis que construir un **modelo honesto "al diagnóstico"** y **demostrar la fuga**.
* **Codificación.** Las **16 variables son texto**; hay que distinguir **nominales** (one-hot) de **ordinales** de estadificación (`T`, `N`, `Stage`, `Risk`), respetando su orden.

Al terminar, debéis saber **razonar la disponibilidad temporal** de cada variable, **codificar** correctamente, comparar y optimizar modelos, tratar el **desequilibrio**, **combinar** si aporta, **interpretar** (SHAP) y **desplegar**.

## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto procede de un **estudio retrospectivo de cohorte** sobre recurrencia de
**cáncer diferenciado de tiroides** (DTC), depositado en el UCI Machine Learning Repository
(*id 915*; Borzooei & Tarokhian, 2023, DOI `10.24432/C5632J`). Los datos se recogieron a lo
largo de **15 años**, con un **seguimiento mínimo de 10 años por paciente**.

Contiene **383 pacientes** (312 mujeres / 71 hombres), **16 variables predictoras** y una
variable objetivo. **No hay valores faltantes** y **no viene pre-dividido** en `train`/`test`.
De las 16 predictoras, **solo `Age` es numérica**; las **15 restantes son cadenas de texto**
y requieren codificación.

Cada fila corresponde a **un paciente**, y combina tres tipos de información de naturaleza muy
distinta: quién es el paciente, **qué se observó en el momento del diagnóstico** (clínica,
patología, estadificación) y **qué ocurrió después del tratamiento** (respuesta terapéutica).
Esa mezcla de lo *previo* y lo *posterior* es la clave para entender qué se puede y qué no se
puede concluir con estos datos.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Perfil del paciente y hábitos**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `Age` | Numérica | 15–82 (media 40,9; DE 15,1) | **Edad al diagnóstico**. Distribución asimétrica a la derecha. |
| `Gender` | Nominal | F / M | **Sexo**. Fuerte desequilibrio: 81 % mujeres. |
| `Smoking` | Binaria | Yes / No | Fumador **en el momento** del diagnóstico. |
| `Hx Smoking` | Binaria | Yes / No | **Historia** de tabaquismo (ex-fumador). |
| `Hx Radiothreapy` | Binaria | Yes / No | Antecedente de **radioterapia** (nótese la errata del original). |

**Bloque 2 — Variables clínico-patológicas** *(observadas al diagnóstico)*

| Variable | Tipo | Descripción |
|---|---|---|
| `Thyroid Function` | Nominal (5) | **Función tiroidea**: eutiroidismo, hiper/hipotiroidismo clínico y subclínico. |
| `Physical Examination` | Nominal (5) | **Exploración física**: bocio uninodular izq./dcho., multinodular, difuso, normal. |
| `Adenopathy` | Nominal (6) | **Adenopatías**: No / Right / Left / Bilateral / Extensive / Posterior. |
| `Pathology` | Nominal (4) | **Tipo histológico**: Papillary / Micropapillary / Follicular / *Hurthel cell*. Muy desbalanceado hacia el papilar. |
| `Focality` | Binaria | **Focalidad**: Uni-Focal / Multi-Focal. |
| `Risk` | **Ordinal** | **Riesgo ATA**: Low < Intermediate < High. **Es una variable derivada** (véase advertencia 2). |

**Bloque 3 — Estadificación TNM (AJCC)** *(derivada de las anteriores)*

| Variable | Tipo | Valores |
|---|---|---|
| `T` | **Ordinal** | T1a < T1b < T2 < T3a < T3b < T4a < T4b (**tamaño/extensión del tumor**) |
| `N` | **Ordinal** | N0 < N1a < N1b (**afectación ganglionar**) |
| `M` | Binaria | M0 / M1 (**metástasis a distancia**; M1 es muy poco frecuente) |
| `Stage` | **Ordinal** | I < II < III < IVA < IVB (**estadio global**) |

**Bloque 4 — Respuesta al tratamiento** *(medida DESPUÉS del tratamiento)*

| Variable | Tipo | Valores |
|---|---|---|
| `Response` | Nominal (4) | Excellent / Indeterminate / Biochemical Incomplete / **Structural Incomplete** |

> ⚠️ **`Response` no existe en el momento del diagnóstico.** Se evalúa tras la cirugía y el
> tratamiento con yodo radiactivo. Si entra en el modelo, el problema deja de ser predicción
> temprana. Es el aviso más importante del preprocesado.

**Variable objetivo**

| Variable | Valores | Descripción |
|---|---|---|
| `Recurred` | Yes / No | **Recurrencia** del tumor durante el seguimiento. Reparto **moderadamente desequilibrado**: 275 No / 108 Yes (**28,2 %** de recurrencias). |

> **Limitación del dataset:** proviene de **un único centro** (Hamadan, Irán) y no incluye
> tratamiento recibido, dosis de yodo, marcadores moleculares (BRAF, TERT), niveles de
> tiroglobulina ni **tiempo hasta la recurrencia**. Tenlo en cuenta al interpretar.
> **Verificad los nombres reales** de las columnas del CSV antes de codificar: hay erratas en
> el original (`Hx Radiothreapy`, `Hurthel cell`).

### <font color="steelblue">Advertencias metodológicas</font>

1. **Fuga temporal: `Response` mira al futuro.** Se recoge después del tratamiento, es decir,
   **después** del punto en el que un clínico querría usar el modelo. Además es el predictor
   dominante: `Structural Incomplete` está tan asociada a la recurrencia que **separa casi
   perfectamente** las clases, y cualquier modelo que la incluya alcanzará métricas
   espectaculares sin haber aprendido nada útil. Decidid **explícitamente** qué escenario
   modeláis —*prequirúrgico* (sin `Response`) o *post-respuesta* (con ella)— y comparad ambos:
   la caída de rendimiento entre uno y otro **es un resultado del proyecto**.

2. **Redundancia estructural: `Stage` y `Risk` no son observaciones, son funciones.**
   `Stage` es un **compuesto determinista** de `T`, `N`, `M` y la edad; `Risk` (ATA) se deriva
   de la histología y el TNM. Es decir, no aportan información nueva: la **recodifican**. Peor
   aún, el riesgo ATA está diseñado precisamente **para predecir la recurrencia**, de modo que
   usarlo como predictor de `Recurred` es en parte **circular**. La literatura suele retener
   solo **13 de las 16** variables tras excluir por derivación. Si las conserváis, no leáis sus
   importancias como si fueran hallazgos.

3. **La codificación ordinal hay que hacerla a mano.** `T`, `N`, `Stage` y `Risk` tienen un
   orden **clínico** que `OrdinalEncoder` no conoce: por defecto ordena **alfabéticamente** y
   os romperá la escala (colocará `High < Intermediate < Low`, y aunque `IVA < IVB` sale bien,
   `III < IVA` no). Definid los `categories` explícitamente. Es el error que más veces se cuela.

4. **No hay partición predefinida y la muestra es pequeña.** Con 383 filas y solo **108 casos
   positivos**, una única partición 80/20 deja ~22 recurrencias en test: el intervalo de
   confianza de cualquier métrica será enorme. Usad **validación cruzada estratificada
   repetida** (p. ej. 5×10) y reportad la **desviación entre folds**, no un número aislado.

5. **Desequilibrio moderado: la exactitud engaña.** Un clasificador que prediga siempre «No»
   acierta el 71,8 %. Reportad **AUC-PR, sensibilidad y F1 sobre la clase minoritaria**, no
   *accuracy*. Cuidado con SMOTE: casi todas las variables son categóricas y la interpolación
   euclídea genera pacientes **clínicamente imposibles** (p. ej. `T1a` con `M1`); si remuestreáis,
   usad SMOTE-NC o pesos de clase.

6. **Categorías raras y separación cuasi-perfecta.** `M1`, `T4b`, `Stage IVB`, `Follicular` y
   `Hurthel cell` tienen muy pocos casos. En regresión logística producen **separación
   completa** (coeficientes que divergen); en validación cruzada, folds **sin ningún ejemplo**
   de la categoría. Agrupad niveles poco frecuentes (`T4a`+`T4b` → `T4`; `Stage IVA`+`IVB` → `IV`)
   **antes** de partir, y documentad la agrupación.

7. **Validez externa limitada.** Un solo centro, un solo país, **81 % de mujeres** y predominio
   casi absoluto del carcinoma papilar. El modelo aprenderá la epidemiología **de esta cohorte**.
   No extrapoléis a poblaciones con distinta distribución histológica o distinto protocolo de
   seguimiento, y no lo presentéis como una herramienta clínica.

8. **Retrospectivo: las variables se extrajeron de la historia clínica.** `Smoking` y
   `Hx Smoking` son **autoinformadas**; `Physical Examination` y `Adenopathy` dependen del
   **observador**. Hay variabilidad interobservador no medible aquí, y los propios autores del
   dataset señalan que la clasificación de riesgo convencional está sujeta a ella. Las
   importancias no son efectos causales: que `Risk` tenga un SHAP alto **no demuestra** que
   intervenir sobre el riesgo reduzca la recurrencia (de hecho, no es una variable sobre la que
   se pueda intervenir).

9. **Consideraciones éticas.** `Age` y `Gender` son **atributos sensibles**, y aquí además están
   *dentro* del criterio clínico: la edad forma parte de la fórmula del estadio AJCC. Con solo
   **71 hombres**, cualquier evaluación de equidad por sexo tendrá intervalos de confianza
   inutilizables. Evaluad el rendimiento **por subgrupos** de todos modos, pero reportad la
   incertidumbre: un modelo de recurrencia oncológica que falle sistemáticamente en un subgrupo
   no es un problema de métrica, es un problema de daño.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Clasificad cada variable por su disponibilidad temporal:** ¿se conoce **al diagnóstico/estadificación** o **tras el tratamiento**? `Response` es post-tratamiento → **excluir** del modelo principal. Razonad también el papel de `Risk` (puntuación clínica derivada).
2. **Codificad bien:** **ordinales** (`T`, `N`, `Stage`, `Risk`) con orden explícito; **nominales** (`Gender`, `Thyroid Function`, `Physical Examination`, `Adenopathy`, `Pathology`) con **one-hot**; **binarias** a 0/1; **`Age`** escalada para modelos de distancia/lineales. Todo en un **`Pipeline`**.
3. **Partición estratificada**; el *test* solo se toca al final.
4. **Equilibrado solo en *train*** (material 11); desequilibrio moderado.
5. **Coste clínico:** no detectar una recurrencia (falso negativo) es lo más grave; vigilad la **sensibilidad/recall**.
6. **Demostrad la fuga:** comparad el modelo honesto con uno que incluya `Response` y explicad la diferencia.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reportad lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Descarga y carga (equivalente a usar %cd $path y leer el CSV)
path = kagglehub.dataset_download("joebeachcapital/differentiated-thyroid-cancer-recurrence")
print("Ruta:", path, "| Archivos:", os.listdir(path))
thyroid = pd.read_csv(os.path.join(path, "Thyroid_Diff.csv"))
print(f"Dimensiones: {thyroid.shape[0]:,} filas × {thyroid.shape[1]} columnas")
thyroid.head()

# <font color="steelblue">Fase 1 — Comprensión, EDA y disponibilidad temporal</font>

**Tareas obligatorias**
1. **Columnas y tipos.** Listad los nombres reales y los **valores únicos** de cada variable. Clasificad cada una en **binaria / nominal / ordinal** (y `Age` numérica).
2. **Disponibilidad temporal (clave).** Para cada variable, decidid si está disponible **al diagnóstico/estadificación** o **solo tras el tratamiento**. Marcad **`Response`** como post-tratamiento y discutid `Risk` (score clínico derivado, disponible pronto pero "resumen del juicio experto").
3. **Objetivo.** Distribución de `Recurred` (confirmad el ~28 % y el desequilibrio moderado).
4. **Relación con el objetivo.** Tasa de recurrencia por `Risk`, `Stage`, `N`, `Adenopathy`, `Response`… (veréis que `Response`/`Structural Incomplete` casi "predice" la recurrencia — esa es la pista de la fuga).
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** ¿por qué incluir `Response` daría un modelo "perfecto" pero **inútil** para la predicción temprana?

# <font color="steelblue">Fase 2 — Codificación, conjunto "al diagnóstico" y partición</font>

**2A. Codificación (clave) — define los grupos de columnas**
1. **Ordinales con orden explícito:** `T`, `N`, `Stage`, `Risk` (usad `OrdinalEncoder(categories=[...])` con el orden clínico).
2. **Nominales con one-hot:** `Gender`, `Thyroid Function`, `Physical Examination`, `Adenopathy`, `Pathology`.
3. **Binarias a 0/1:** `Smoking`, `Hx Smoking`, `Hx Radiotherapy`, `Focality`, `M`, y el objetivo `Recurred`.
4. **`Age`** → escalar para logística/SVM/kNN (los árboles no).

**2B. Conjunto de variables "al diagnóstico" (clave)**
5. Definid `X_diag` **excluyendo `Response`** (y razonad si excluís también `Risk`). Guardad aparte `Response` para la **demostración de la fuga** (Fase 7).

**2C. Partición y `Pipeline`**
6. **Partición estratificada** y `ColumnTransformer`/`Pipeline` que aplique a cada grupo su transformación.

> **A responder:** justificad cada decisión de codificación (¿por qué `Stage` es ordinal y `Pathology` nominal?).

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias** (con el conjunto **al diagnóstico**, sin `Response`)
1. Comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación cruzada repetida** estratificada (n pequeño) con métrica adecuada (**ROC-AUC**, **recall** de recurrencia, F1).
3. **Tabla** comparativa y comentario.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

Con ~28 % de recurrencias el desequilibrio es **moderado**; es obligatorio **medir** el efecto de tratarlo (material **11**) sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (penaliza más no detectar recurrencias).
3. **Sobremuestreo:** **SMOTE** (numéricas/codificadas; dentro del `ImbPipeline`).
4. (Opcional) submuestreo/híbrido.

Reportad **recall de recurrencia**, **F1**, **ROC-AUC** y exactitud balanceada, y razonad la mejor opción.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y la métrica elegida; búsqueda **sobre el `Pipeline`** (prefijo `clf__`).
3. (Recomendado por el n pequeño) **CV anidada** para estimación honesta.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** (ROC-AUC/recall): ¿mejora? ¿compensa el coste/menor interpretabilidad?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, demostración de la fuga e interpretación</font>

Esta es la fase **distintiva**. El *test* se usa una sola vez.

**Tareas obligatorias**
1. **Modelo honesto (al diagnóstico).** Evaluadlo en el *test*: **matriz de confusión**, **recall de recurrencia**, **especificidad**, **F1**, **ROC-AUC**, **PR-AUC**.
2. **Demostración de la fuga (clave).** Entrenad un **segundo** modelo **añadiendo `Response`** como predictora y comparad: veréis que la *accuracy*/AUC sube a ~**0.97–0.99** (como en la literatura). **Explicad por qué** ese salto es **fuga temporal** y por qué el modelo honesto, aunque dé números más bajos, es el **clínicamente útil** para el cribado **temprano**.
3. **Interpretabilidad (SHAP).** En el modelo con `Response`, SHAP mostrará que `Response` (Structural Incomplete) **domina** — esa es la prueba de la fuga. En el honesto, ¿mandan `N`/`Adenopathy`/`Stage`/`Risk`, como en la clínica?
4. **(Opcional) Benchmark `Risk` (ATA).** Comparad vuestro modelo con la regla `Risk = High ⇒ recurrencia` (igual que el BI-RADS del caso de mamografía).
5. **Discusión crítica:** un solo centro, tamaño muestral, qué significa una recurrencia no detectada.

# <font color="steelblue">Fase 8 — Despliegue del modelo (al diagnóstico)</font>

1. **Persistencia:** guardad el **`Pipeline` completo** del **modelo honesto** (sin `Response`) con `joblib`.
2. **Función de predicción:** `predecir_recurrencia(...)` con las variables disponibles **al diagnóstico** (demografía, clínico-patológicas, TNM) que devuelva clase y **probabilidad**.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con selectores para las variables categóricas y la edad; salida = riesgo de recurrencia. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**.

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa** de apoyo a la decisión; **no** sustituye la valoración médica ni los protocolos ATA/AJCC.

# <font color="steelblue">Pistas y errores típicos</font>

* **El ~99 % es una trampa.** La literatura lo alcanza apoyándose en `Response`, que se conoce **después** del tratamiento. Para predicción **temprana**, esa variable es **fuga**: exclúyela y demuéstralo.
* **`Risk` es un score clínico.** Disponible pronto, pero "resume el juicio experto" (como el BI-RADS): úsalo con cautela y considéralo también como **benchmark**.
* **Ordinal vs nominal.** `T`, `N`, `Stage`, `Risk` tienen orden (codifícalos con `OrdinalEncoder` y el orden explícito); `Pathology`, `Adenopathy`, etc. son nominales (one-hot).
* **n pequeño (383):** usa **CV repetida** y, al optimizar, **CV anidada**; no te fíes de un único *split*.
* **Coste clínico:** prioriza el **recall de recurrencia** (no perder recaídas).
* **Despliegue:** guarda el **Pipeline entero** del modelo **honesto** y respeta el orden/formato de las columnas.

# <font color="steelblue">Referencias</font>

* Borzooei, S., Briganti, G., Golparian, M., Lechien, J. R. & Tarokhian, A. (2024). *Machine learning for risk stratification of thyroid cancer patients: a 15-year cohort study*. Eur. Arch. Oto-Rhino-Laryngol., 281, 2095–2104.
* Borzooei, S. & Tarokhian, A. (2023). *Differentiated Thyroid Cancer Recurrence*. UCI ML Repository (id 915).
* *Explaining Risk Stratification in Differentiated Thyroid Cancer Using SHAP and ML*. MDPI Biomedicines, 2025.
* Haugen, B. R. et al. (2016). *2015 ATA Management Guidelines for DTC*. Thyroid, 26(1).
* Cuadernos del curso: *Boosting*, *Random Forest*, *SVM*, *Regresión logística binaria*, *Equilibrando las muestras*.